In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [7]:
df = pd.read_csv('cleaned_data.csv')
df

,name,company,year,Price,kms_driven,fuel_type,kms_group
0,Santro Xing,Hyundai,2007,80000.0,45000.0,Petrol,25k-50k
1,Grand i10,Hyundai,2014,325000.0,28000.0,Petrol,25k-50k
2,EcoSport Titanium,Ford,2014,575000.0,36000.0,Diesel,25k-50k
3,Figo,Ford,2012,175000.0,41000.0,Diesel,25k-50k
4,Eon,Hyundai,2013,190000.0,25000.0,Petrol,0-25k
...,...,...,...,...,...,...,...
699,Suzuki Ritz,Maruti,2011,270000.0,50000.0,Petrol,25k-50k
700,Indica V2,Tata,2009,110000.0,30000.0,Diesel,25k-50k
701,Corolla Altis,Toyota,2009,300000.0,132000.0,Petrol,100k-150k
702,Zest XM,Tata,2018,260000.0,27000.0,Diesel,25k-50k


In [4]:
df = df.drop('kms_group', axis=1)

In [6]:
df

,name,company,year,Price,kms_driven,fuel_type
0,Santro Xing,Hyundai,2007,80000.0,45000.0,Petrol
1,Grand i10,Hyundai,2014,325000.0,28000.0,Petrol
2,EcoSport Titanium,Ford,2014,575000.0,36000.0,Diesel
3,Figo,Ford,2012,175000.0,41000.0,Diesel
4,Eon,Hyundai,2013,190000.0,25000.0,Petrol
...,...,...,...,...,...,...
699,Suzuki Ritz,Maruti,2011,270000.0,50000.0,Petrol
700,Indica V2,Tata,2009,110000.0,30000.0,Diesel
701,Corolla Altis,Toyota,2009,300000.0,132000.0,Petrol
702,Zest XM,Tata,2018,260000.0,27000.0,Diesel


In [8]:
#Lets divide dataset into X and y
X = df[['company', 'name', 'year', 'kms_driven', 'fuel_type']]
y = df[['Price']]

In [9]:
y

,Price
0,80000.0
1,325000.0
2,575000.0
3,175000.0
4,190000.0
...,...
699,270000.0
700,110000.0
701,300000.0
702,260000.0


In [10]:
X

,company,name,year,kms_driven,fuel_type
0,Hyundai,Santro Xing,2007,45000.0,Petrol
1,Hyundai,Grand i10,2014,28000.0,Petrol
2,Ford,EcoSport Titanium,2014,36000.0,Diesel
3,Ford,Figo,2012,41000.0,Diesel
4,Hyundai,Eon,2013,25000.0,Petrol
...,...,...,...,...,...
699,Maruti,Suzuki Ritz,2011,50000.0,Petrol
700,Tata,Indica V2,2009,30000.0,Diesel
701,Toyota,Corolla Altis,2009,132000.0,Petrol
702,Tata,Zest XM,2018,27000.0,Diesel


In [11]:
#Lets create pipeline which consists of columntransformer - whose job is to give numerical columns
#for string columns using OneHotEncoder and it returns other numerical columns as it is.
#After column transformer we create model object and put it in pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

ohe = OneHotEncoder()
ohe.fit(X[["company", "name", "fuel_type"]])
ct = make_column_transformer((OneHotEncoder(categories = ohe.categories_), 
                              ["company", "name", "fuel_type"]), remainder = 'passthrough')
model = LinearRegression()

pipe = make_pipeline(ct, model)

In [12]:
pipe

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
#Lets divide dataset into training and testing

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

scores = []

for i in range(0, 101):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.05, random_state = i)
    #Lets train pipeline
    pipe.fit(X_train, y_train)
    #Lets find r2_score
    y_pred = pipe.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
    result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
    score = r2_score(result['Price'], result['Prediction'])
    scores.append(score)
    

In [17]:
scores

[0.07743739585099207,
 0.6379298926217596,
 0.4919875682051307,
 0.7143787550003242,
 -0.23074902353390936,
 0.6851330861692644,
 -0.1066811413181088,
 0.7715465644575381,
 0.39771882029371175,
 0.6471645499723689,
 0.6822413378991283,
 0.2485460472830091,
 0.675712606602352,
 -1.1182750531019674,
 0.06978099125817938,
 0.7609055133128464,
 0.5171914767688328,
 0.6462739068994905,
 0.4991040767859438,
 0.86007846217071,
 0.3197869996482642,
 0.06726511635223609,
 0.5180600231751227,
 0.5589550257222548,
 0.7791753129945714,
 0.5632467252457198,
 0.8200623201810784,
 0.13065038776478366,
 0.7678489970347744,
 0.3440882346250246,
 -0.2817444289776718,
 0.568349434085322,
 0.44776503923922095,
 0.7266335846470047,
 0.6856400014405784,
 0.7631513889840933,
 0.7933385078591885,
 0.5872285426308181,
 0.21883122315659742,
 0.5735707080591103,
 0.06455836942805393,
 0.21220055262100324,
 0.7386497423979876,
 -0.2672715235353931,
 -0.9252303683456273,
 -0.20684568209484722,
 0.25872789959075915

In [19]:
#Find index of max score
index = np.argmax(scores)
index

np.int64(19)

In [20]:
#Lets retrain with best index
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.05, random_state = index)
pipe.fit(X_train, y_train)


,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [21]:
data = [['Ford', 'EcoSport', 2015, 80000, 'Petrol']]
columns = ['company', 'name', 'year', 'kms_driven', 'fuel_type']
myinput = pd.DataFrame(data = data, columns = columns)
price = pipe.predict(myinput)
print("You can purchase this car for:", round(price[0,0]))

You can purchase this car for: 547696


In [22]:
#To get input from user and provide output we must develop web application
#We can use Python - Flask, FastAPI, Django or Streamlit
#We cannot train model all the time, so lets export out trained model(pipe) using pickle library
import pickle as pkl
pkl.dump(pipe, open('car-price-predictor.pkl', 'wb'))